# MVP-FAS

https://github.com/sheriff1max/MVP-FAS

In [1]:
def hour2sec(hour: float) -> float:
    return hour * 60 * 60

In [2]:
PARAMS = {
    'domain': 'figure',  # cutout | figure | mannequin | mask | print | screen
    'description': 'MVP-FAS',
    'direction': 'maximize',

    'n_trials': 500,
    'timeout': hour2sec(6),  # нужно в секундах hour2sec(6)

    'w_sim': 0.5,
    'w_prob': 0.5,
}


_domains = ['cutout', 'figure', 'mannequin', 'mask', 'print', 'screen']
PARAMS['exclude_folders'] = [domain for domain in _domains if domain != PARAMS['domain']]
print(f"{PARAMS['exclude_folders'] = }")

PARAMS['description'] += f' domain={PARAMS["domain"]}'
print(f"{PARAMS['description'] = }")

PARAMS['exclude_folders'] = ['cutout', 'mannequin', 'mask', 'print', 'screen']
PARAMS['description'] = 'MVP-FAS domain=figure'


In [3]:
import shutil
# shutil.rmtree('MVP-FAS')
# shutil.rmtree('fas_aug_attack')

In [4]:
!git clone https://github.com/sheriff1max/MVP-FAS.git -q

In [5]:
!git clone https://github.com/sheriff1max/fas_aug_attack.git -q

In [6]:
!pip install yacs -q
!pip install ftfy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00


In [7]:
import sys

path_model = '/kaggle/working/MVP-FAS'
if path_model not in sys.path:
    sys.path.insert(1, path_model)

path_attack = '/kaggle/working/fas_aug_attack'
if path_attack not in sys.path:
    sys.path.insert(1, path_attack)

In [8]:
import os
import random
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
from torch.nn import functional as F
import torchvision

import matplotlib.pyplot as plt

from configs.cfg import _C as cfg

from models.make_network import get_network

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [10]:
def set_pretrained_setting(net, weight_path, device):
    checkpoint_dict = torch.load(weight_path, device, weights_only=False)
    checkpoint = checkpoint_dict['state_dict']
    last_epoch = checkpoint_dict['epoch'] - 1
    
    pretrained_dict = {k: v for k, v in checkpoint.items() if k in net.module.state_dict().keys()}
    net.module.load_state_dict(pretrained_dict)
    return net, last_epoch


def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [11]:
seed = 42

checkpoint = '/kaggle/input/datasets/morph1max/mvp-fas-weights/MVP-FAS_weights/MVP-FAS_P2_weights/drive-download-20260403T091807Z-3-003/MVP_FAS_P2_S.pth'

set_seed(seed, deterministic=False)

net = get_network(cfg, device, net_name='MVP_FAS')
net, last_epoch = set_pretrained_setting(net, checkpoint, device)

100%|████████████████████████████████████████| 335M/335M [00:02<00:00, 173MiB/s]


Finetune layer in backbone: visual.class_embedding
Finetune layer in backbone: visual.positional_embedding
Finetune layer in backbone: visual.proj
Finetune layer in backbone: visual.conv1.weight
Finetune layer in backbone: visual.ln_pre.weight
Finetune layer in backbone: visual.ln_pre.bias
Finetune layer in backbone: visual.transformer.resblocks.0.attn.in_proj_weight
Finetune layer in backbone: visual.transformer.resblocks.0.attn.in_proj_bias
Finetune layer in backbone: visual.transformer.resblocks.0.attn.out_proj.weight
Finetune layer in backbone: visual.transformer.resblocks.0.attn.out_proj.bias
Finetune layer in backbone: visual.transformer.resblocks.0.ln_1.weight
Finetune layer in backbone: visual.transformer.resblocks.0.ln_1.bias
Finetune layer in backbone: visual.transformer.resblocks.0.mlp.c_fc.weight
Finetune layer in backbone: visual.transformer.resblocks.0.mlp.c_fc.bias
Finetune layer in backbone: visual.transformer.resblocks.0.mlp.c_proj.weight
Finetune layer in backbone: vi

# Dataset

In [12]:
from src.utils.dataset import AttackDataset

dataset = AttackDataset(
    path='/kaggle/input/datasets/morph1max/spoof-attack-liveness-face',
    exclude_folders=PARAMS['exclude_folders'],
)
print(f'Returns keys = {dataset[0].keys()}')
len(dataset)

Returns keys = dict_keys(['img', 'filename', 'path2file', 'is_real', 'type_attack'])


42

In [13]:
# plt.imsave('example.png', dataset[300]['img']);

# Model

In [14]:
def get_transform(
    img_size: tuple[int] = (224, 224),
    normalize_mean: list[float] = [0.485, 0.456, 0.406],
    normalize_std: list[float] = [0.229, 0.224, 0.225],
):
    return torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=normalize_mean, std=normalize_std),
            torchvision.transforms.Resize(img_size),
        ]
    )

to_tensor = get_transform()

In [15]:
import torch
import torchvision.models as models
import torchvision.transforms as T
import torch.nn.functional as F


net_sim = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
net_sim = torch.nn.Sequential(*list(net_sim.children())[:-1])
net_sim.eval()


def image_similarity(img1, img2):
    img1 = to_tensor(img1).unsqueeze(0)
    img2 = to_tensor(img2).unsqueeze(0)

    with torch.no_grad():
        emb1 = F.normalize(net_sim(img1), p=2, dim=1)
        emb2 = F.normalize(net_sim(img2), p=2, dim=1)
    
    similarity = F.cosine_similarity(emb1, emb2).item()
    return similarity

# Использование
score = image_similarity(dataset[0]['img'], dataset[0]['img'])
print(f"Схожесть: {score:.4f}")  # 1.0 = идентичные

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 166MB/s]


Схожесть: 1.0000


In [16]:
from src.base import BaseModel
from typing import Any


class MVP_FAS_Model(BaseModel):
    def __init__(self, model: Any):
        super().__init__(model=model)

    def predict(self, img: Any) -> float:
        net.eval()
        with torch.no_grad():
            img = to_tensor(img).unsqueeze(0)
            img = img.to(device)
    
            # forward
            val_results = net(img)
            val_output_list = val_results['similarity']

            # metric
            # spoofing | real
            #     0       1

            # Вероятность положительного класса. Нужно максимизировать.
            score = F.softmax(val_output_list, dim=-1).detach().cpu().numpy()[:, -1][0]
            return float(score)

    
model = MVP_FAS_Model(model=net)
pred = model.predict(dataset[0]['img'])
print(pred, type(pred))

6.167036917759106e-05 <class 'float'>


# Transforms

In [17]:
from src import transforms
from src.base import BaseTransform
import inspect

In [ ]:
list_type_transforms = [
    # Advanced transformations
    [
        transforms.MorphologicalTransform,
        None,
    ],

    # Noise transforms
    [
        transforms.PixelDropoutTransform,
        None,
    ],

    # Color transformations
    [
        transforms.BrightnessContrastTransform,
        transforms.HSVTransform,
        transforms.RGBShiftTransform,
        transforms.GammaTransform,
        transforms.SolarizeTransform,
        transforms.PosterizeTransform,
        transforms.EqualizeTransform,
        transforms.InvertTransform,
        transforms.ToGrayTransform,
        transforms.ChannelShuffleTransform,
        transforms.ToSepiaTransform,
        None,
    ],

    # Transformations of blurring and sharpness
    [
        transforms.BlurTransform,
        transforms.GaussianBlurTransform,
        transforms.MedianBlurTransform,
        transforms.MotionBlurTransform,
        transforms.SharpenTransform,
        transforms.EmbossTransform,
        None,
    ],

    # Weather and atmospheric conditions
    [
        transforms.RainTransform,
        transforms.SnowTransform,
        transforms.RainbowTransform,
        transforms.ChromaticAberrationTransform,
        transforms.DefocusTransform,
        transforms.ZoomBlurTransform,
        None,
    ],

    # Geometric transformations
    [
        transforms.PerspectiveTransform,
        transforms.OpticalDistortionTransform,
        transforms.ShiftScaleRotateTransform,
        None,
    ],

    # Transformation of compression and artifacts
    [
        transforms.CompressionTransform,
        transforms.DownscaleTransform,
        None,
    ],

    # Dropout transforms
    [
        transforms.CoarseDropoutTransform,
        transforms.GridDropoutTransform,
        None,
    ],
]


list_type_transforms[:5]

[[src.transforms.transforms.MorphologicalTransform, None],
 [src.transforms.transforms.PixelDropoutTransform, None],
 [src.transforms.transforms.BrightnessContrastTransform,
  src.transforms.transforms.HSVTransform,
  src.transforms.transforms.RGBShiftTransform,
  src.transforms.transforms.GammaTransform,
  src.transforms.transforms.SolarizeTransform,
  src.transforms.transforms.PosterizeTransform,
  src.transforms.transforms.EqualizeTransform,
  src.transforms.transforms.InvertTransform,
  src.transforms.transforms.ToGrayTransform,
  src.transforms.transforms.ChannelShuffleTransform,
  src.transforms.transforms.ToSepiaTransform,
  None],
 [src.transforms.transforms.BlurTransform,
  src.transforms.transforms.GaussianBlurTransform,
  src.transforms.transforms.MedianBlurTransform,
  src.transforms.transforms.MotionBlurTransform,
  src.transforms.transforms.SharpenTransform,
  src.transforms.transforms.EmbossTransform,
  None],
 [src.transforms.transforms.RainTransform,
  src.transforms.t

# Pipeline

In [20]:
from src.pipeline import PipelineAttackOptunaDataset, PipelineAttackOptunaImg
from src.utils.logging import LoggerOptuna

In [21]:
from src.base import BasePipelineAttackOptuna
from src.pipeline import PipelineAttackImg
from src.utils.utils import make_list_transforms_optuna
from typing import Type
import optuna
import pandas as pd


class PipelineAttackWithSimilarityOptunaDataset(BasePipelineAttackOptuna):

    def __init__(
        self,
        model: BaseModel,
        list_type_transforms: list[list[Type[BaseTransform] | None]],
        w_prob: float = 0.5,
        w_sim: float = 0.5,
        logger: LoggerOptuna = None,
    ):
        super().__init__(
            model=model,
            list_type_transforms=list_type_transforms,
            logger=logger,
        )
        self.w_prob = w_prob
        self.w_sim = w_sim
        self._data = None

    def _objective(self, trial: optuna.Trial) -> float:
        """Целевая функция для оптимизации

        :param trial: optuna для оптимизации
        :return: уверенность модели
        """
        list_transforms = make_list_transforms_optuna(
            trial=trial,
            list_type_transforms=self.list_type_transforms,
        )

        attack_pipeline = PipelineAttackImg(
            model=self.model,
            list_transforms=list_transforms,
        )

        list_probs = []
        list_sims = []
        list_scores = []
        for i in range(len(self._data)):
            img = self._data[i]['img']
            response = attack_pipeline.attack(img)

            score_sim = image_similarity(img, response.img)
            score_prob = response.score
            # print(f'{score_sim = } | {score_prob = }')

            score = self.w_prob * score_prob + self.w_sim * score_sim

            list_probs.append(score_prob)
            list_sims.append(score_sim)
            list_scores.append(score)

        prob = np.mean(list_probs)
        sim = np.mean(list_sims)
        score = np.mean(list_scores)


        with open('/kaggle/working/logs/run_1/tmp_scores.txt', 'a') as f:
            f.write(f'{prob}|{sim}|{score}\n')
        
        if self.logger:
            self.logger.step(
                img=response.img,
                score=score,
                step=trial.number,
                params=trial.params,
            )
        return score

    def baseline(self, data: Any) -> float:
        """"""
        attack_pipeline = PipelineAttackImg(
            model=self.model,
            list_transforms=[],
        )

        # Подсчёт score на исходных изображениях без преобразований.
        list_scores = []
        for i in range(len(data)):
            img = data[i]['img']
            response = attack_pipeline.attack(img)
            score = response.score

            list_scores.append(score)

        score = np.mean(list_scores)
        self.logger.save_json(
            data={'score': score, 'description': 'mean by dataset'},
            filename='baseline_score.json',
        )

        return score

In [22]:
logger = LoggerOptuna(
    direction=PARAMS['direction'],
    description=PARAMS['description'],
)

# DATASET WITH SIMILARITY
optuna_attack_pipeline_dataset_with_sim = PipelineAttackWithSimilarityOptunaDataset(
    model=model,
    list_type_transforms=list_type_transforms,
    logger=logger,
    w_prob=PARAMS['w_prob'],
    w_sim=PARAMS['w_sim'],
)

optuna_attack_pipeline_dataset_with_sim.optimize(
    data=dataset,
    direction=PARAMS['direction'],
    n_trials=PARAMS['n_trials'],
    timeout=PARAMS['timeout'],
    show_progress=True,
    catch=(ValueError, ),
)

# DATASET
# optuna_attack_pipeline_dataset = PipelineAttackOptunaDataset(
#     model=model,
#     list_type_transforms=list_type_transforms,
#     logger=logger,
# )

# optuna_attack_pipeline_dataset.optimize(
#     data=dataset,
#     direction=PARAMS['direction'],
#     n_trials=PARAMS['n_trials'],
#     timeout=PARAMS['timeout'],
#     show_progress=True,
#     catch=(ValueError, ),
# )

# IMG
# optuna_attack_pipeline_img = PipelineAttackOptunaImg(
#     model=model,
#     list_type_transforms=list_type_transforms,
#     logger=logger,
# )

# optuna_attack_pipeline_img.optimize(
#     data=dataset[0]['img'],
#     direction=PARAMS['direction'],
#     n_trials=PARAMS['n_trials'],
#     timeout=PARAMS['timeout'],
#     show_progress=True,
#     catch=(ValueError, ),
# )

  0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/albumentations/augmentations/blur/functional.py:232: UserWarning: blur_limit: Non-zero kernel sizes must be odd. Range (4, 5) automatically adjusted to (5, 5).
  result = _ensure_odd_values(result, info.field_name)
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.12/dist-packages/albumentations/augmentations/blur/functional.py:232: UserWarning: blur_limit: Non-zero kernel sizes must be odd. Range (7, 8) automatically adjusted to (7, 9).
  result = _ensure_odd_values(result, info.field_name)
/usr/local/lib/python3.12/dist-packages/albumentations/augmentations/blur/functional.py:232: UserWarning: blur_limit: Non-zero kernel sizes must be odd. Range (3, 4) automatically adjusted to (3, 5).
  result = _ensure_odd_values(result, info.field_name)
/usr/loc

In [23]:
print(os.listdir('/kaggle/working/logs'))

['run_1']


In [24]:
# !zip -FSr run.zip /kaggle/working/logs/run_1

In [25]:
shutil.rmtree('MVP-FAS')
shutil.rmtree('fas_aug_attack')